In [19]:
import json
import re
import random
from pathlib import Path
from collections import Counter
 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
 
sns.set_style("whitegrid")
 
DATA_PATH = Path(r"D:\J\Desktop\language_technology\course\projects_AI\mt_oil_no\data\source\npd\npd_training_mt.json")
OUT_DIR   = Path("reports/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)
 
data = json.loads(DATA_PATH.read_text(encoding='utf-8'))
print(f"Loaded {len(data)} pairs")

Loaded 26324 pairs


In [20]:
def word_lengths(data):
    src = [len(d['source'].split()) for d in data]
    tgt = [len(d['target'].split()) for d in data]
    rat = [t / s if s > 0 else 0 for s, t in zip(src, tgt)]
    return src, tgt, rat
 
def compute_oqs(data):
    src = [d['source'] for d in data]
    tgt = [d['target'] for d in data]
    pairs = list(zip(src, tgt))
 
    _, _, ratios = word_lengths(data)
 
    TERMS = ['oil','gas','drilling','well','production','field','reservoir',
             'offshore','pipeline','crude','petroleum','exploration','extraction']
    corpus = ' '.join(src).lower()
 
    components = {
        'alignment':        sum(0.5 <= r <= 2.0 for r in ratios) / len(ratios),
        'completeness':     sum(bool(d['source'].strip() and d['target'].strip()) for d in data) / len(data),
        'src_uniqueness':   1 - len({s for s in src if src.count(s) > 1}) / len(set(src)),
        'tgt_uniqueness':   1 - len({t for t in tgt if tgt.count(t) > 1}) / len(set(tgt)),
        'pair_uniqueness':  1 - len({p for p in pairs if pairs.count(p) > 1}) / len(set(pairs)),
        'domain_relevance': sum(t in corpus for t in TERMS) / len(TERMS),
    }
    oqs = sum(components.values()) / len(components)
    return oqs, components

In [21]:
src_len, tgt_len, ratios = word_lengths(data)
 
print(f"  src  mean={np.mean(src_len):.1f}  median={np.median(src_len):.1f}")
print(f"  tgt  mean={np.mean(tgt_len):.1f}  median={np.median(tgt_len):.1f}")
print(f"  ratio mean={np.mean(ratios):.2f}")

  src  mean=13.4  median=12.0
  tgt  mean=11.2  median=10.0
  ratio mean=0.87


In [22]:
oqs_before, comp_before = compute_oqs(data)
 
print("\n── QC before ──")
for k, v in comp_before.items():
    print(f"  {k:<20} {v:.3f}")
grade = "EXCELLENT" if oqs_before >= 0.91 else "GOOD" if oqs_before >= 0.80 else "POOR"
print(f"  {'OQS':<20} {oqs_before:.3f}  [{grade}]")
 
if oqs_before < 0.80:
    raise RuntimeError("OQS below threshold — inspect data before proceeding")


── QC before ──
  alignment            0.989
  completeness         1.000
  src_uniqueness       0.935
  tgt_uniqueness       0.925
  pair_uniqueness      0.939
  domain_relevance     1.000
  OQS                  0.965  [EXCELLENT]


In [23]:
import re
def remove_duplicates(data):
    seen_pairs = set()
    seen_sources = set()
    unique_data = []
    
    for item in data:
        source = item['source'].strip()
        target = item['target'].strip()
        pair_key = f"{source}|||{target}"
        
        if pair_key not in seen_pairs and source not in seen_sources:
            seen_pairs.add(pair_key)
            seen_sources.add(source)
            unique_data.append(item)
    
    return unique_data

def filter_anomalies(data):
    filtered_data = []
    
    for item in data:
        source_len = len(item['source'].split())
        target_len = len(item['target'].split())
        
        if source_len == 0 or target_len == 0:
            continue
            
        ratio = target_len / source_len
        
        if 0.1 <= ratio <= 10.0 and source_len >= 2 and target_len >= 2:
            filtered_data.append(item)
    
    return filtered_data

def basic_text_cleaning(data):
    cleaned_data = []
    
    for item in data:
        source = re.sub(r'\s+', ' ', item['source'].strip())
        target = re.sub(r'\s+', ' ', item['target'].strip())
        
        if len(source) > 0 and len(target) > 0:
            cleaned_item = item.copy()
            cleaned_item['source'] = source
            cleaned_item['target'] = target
            cleaned_data.append(cleaned_item)
    
    return cleaned_data

print(f"Original dataset size: {len(data)}")

step1_data = remove_duplicates(data)
print(f"After duplicate removal: {len(step1_data)} (-{len(data)-len(step1_data)})")

step2_data = filter_anomalies(step1_data)
print(f"After anomaly filtering: {len(step2_data)} (-{len(step1_data)-len(step2_data)})")

step3_data = basic_text_cleaning(step2_data)
print(f"After text cleaning: {len(step3_data)} (-{len(step2_data)-len(step3_data)})")

cleaned_ratios = []
for item in step3_data:
    source_len = len(item['source'].split())
    target_len = len(item['target'].split())
    cleaned_ratios.append(target_len / source_len)

new_alignment_quality = sum(1 for r in cleaned_ratios if 0.5 <= r <= 2.0) / len(cleaned_ratios)
print(f"New alignment quality: {new_alignment_quality:.3f}")

output_path = r"D:\J\Desktop\language_technology\course\projects_AI\mt_oil_no\data\processed\npd_splits\npd_no_en_train_cleaned.jsonl"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(step3_data, f, ensure_ascii=False, indent=2)

print(f"Cleaned dataset saved to: {output_path}")
print("Ready for train/val/test split")

Original dataset size: 26324
After duplicate removal: 18197 (-8127)
After anomaly filtering: 17435 (-762)
After text cleaning: 17435 (-0)
New alignment quality: 0.987
Cleaned dataset saved to: D:\J\Desktop\language_technology\course\projects_AI\mt_oil_no\data\processed\npd_splits\npd_no_en_train_cleaned.jsonl
Ready for train/val/test split


In [24]:
oqs_after, comp_after = compute_oqs(data_clean)
 
print("\n── QC after ──")
header = f"  {'metric':<20} {'before':>8}  {'after':>8}"
print(header)
print("  " + "-" * (len(header) - 2))
for k in comp_before:
    print(f"  {k:<20} {comp_before[k]:8.3f}  {comp_after[k]:8.3f}")
print(f"  {'OQS':<20} {oqs_before:8.3f}  {oqs_after:8.3f}")
 
cleaned_path = DATA_PATH.parent / "cleaned_dataset_npd.json"
cleaned_path.write_text(json.dumps(data_clean, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"\nSaved cleaned data → {cleaned_path}")


── QC after ──
  metric                 before     after
  ---------------------------------------
  alignment               0.989     0.987
  completeness            1.000     1.000
  src_uniqueness          0.935     1.000
  tgt_uniqueness          0.925     0.989
  pair_uniqueness         0.939     1.000
  domain_relevance        1.000     0.769
  OQS                     0.965     0.958

Saved cleaned data → D:\J\Desktop\language_technology\course\projects_AI\mt_oil_no\data\source\npd\cleaned_dataset_npd.json
